# The Shape of Discovery — Colab Environment Setup

Run this notebook first in every Colab session. It clones the private repo, installs dependencies, and restores any previously saved results from Google Drive.

**First time only:** You need to add your GitHub token as a Colab secret.
1. Click the 🔑 key icon in the left sidebar
2. Add a secret named `GITHUB_TOKEN`
3. Paste your fine-grained personal access token (from emergent-inquiry account)
4. Toggle "Notebook access" ON

## Step 1: Mount Google Drive
Results persist here between sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create project directory on Drive if it doesn't exist
!mkdir -p /content/drive/MyDrive/shape-of-discovery/data
!mkdir -p /content/drive/MyDrive/shape-of-discovery/figures
print('Drive mounted and directories ready.')

## Step 2: Clone the private repo

In [ ]:
import os
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')

# Clone or pull latest
if not os.path.exists('/content/the-shape-of-discovery'):
    !git clone https://{token}@github.com/emergent-inquiry/the-shape-of-discovery.git /content/the-shape-of-discovery
    print('Repo cloned.')
else:
    %cd /content/the-shape-of-discovery
    !git pull
    print('Repo updated.')

%cd /content/the-shape-of-discovery

# Remove token from git remote to prevent leaking credentials
!git remote set-url origin https://github.com/emergent-inquiry/the-shape-of-discovery.git
print('Remote sanitized (token removed).')

## Step 3: Install dependencies

In [ ]:
# Install build dependencies for ripser (needs Cython and C++ compiler)
!pip install -q Cython
# Force-reinstall numpy<2.0 to avoid ABI incompatibility with Colab's pre-installed packages
!pip install --force-reinstall "numpy>=1.24,<2.0"
# Install ripser and persim explicitly (may need to build from source)
!pip install ripser persim
# Install pyflagser for directed flag complex (memory-efficient backend)
!pip install pyflagser
# Then install the rest of the requirements
!pip install -q -r requirements.txt
print('Dependencies installed.')

⚠️ **After running the install cell for the first time, restart the runtime** (Runtime → Restart runtime), then **re-run from Step 1** (mount drive, clone repo) but **skip the install cell**. The restart is needed because numpy was force-reinstalled and the old version is still loaded in memory.

## Step 4: Restore previous results from Drive
If you've run notebooks before, this copies saved data and figures back into the working directory.

In [ ]:
import os
import shutil
import glob

drive_data = '/content/drive/MyDrive/shape-of-discovery/data'
drive_figs = '/content/drive/MyDrive/shape-of-discovery/figures'
local_data = '/content/the-shape-of-discovery/data'
local_figs = '/content/the-shape-of-discovery/figures'

# Restore data files
data_files = glob.glob(f'{drive_data}/*')
for f in data_files:
    dest = os.path.join(local_data, os.path.basename(f))
    if not os.path.exists(dest):
        if os.path.isdir(f):
            shutil.copytree(f, dest)
        else:
            shutil.copy2(f, dest)
        print(f'  Restored: {os.path.basename(f)}')

# Restore figures
fig_files = glob.glob(f'{drive_figs}/*')
for f in fig_files:
    dest = os.path.join(local_figs, os.path.basename(f))
    if not os.path.exists(dest):
        shutil.copy2(f, dest)
        print(f'  Restored: {os.path.basename(f)}')

if not data_files and not fig_files:
    print('No previous results to restore. Fresh start.')
else:
    print(f'Restored {len(data_files)} data items and {len(fig_files)} figures.')

## Step 5: Verify environment

In [ ]:
import os
import psutil
import platform

ram_gb = psutil.virtual_memory().total / (1024**3)
cpu_count = psutil.cpu_count()

print(f'Python:    {platform.python_version()}')
print(f'RAM:       {ram_gb:.1f} GB')
print(f'CPUs:      {cpu_count}')
print(f'Working:   {os.getcwd()}')

# GPU detection
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f'GPU:       {gpu_name} ({gpu_mem:.1f} GB)')
    else:
        print('GPU:       None (select A100 via Runtime \u2192 Change runtime type)')
except ImportError:
    gpu_info = os.popen('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits 2>/dev/null').read().strip()
    if gpu_info:
        name, mem = gpu_info.split(', ')
        print(f'GPU:       {name} ({float(mem)/1024:.1f} GB)')
    else:
        print('GPU:       None (select A100 via Runtime \u2192 Change runtime type)')

print()

# Check key packages
packages = ['pandas', 'numpy', 'scipy', 'networkx', 'ripser', 'pyflagser', 'persim', 'sklearn', 'matplotlib']
for pkg in packages:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, '__version__', '?')
        print(f'  {pkg}: {ver}')
    except ImportError:
        print(f'  {pkg}: NOT INSTALLED')

print()

# Check data status
data_files = os.listdir('data')
parquets = [f for f in data_files if f.endswith('.parquet')]
if parquets:
    print(f'Data ready: {len(parquets)} parquet files found')
    for f in sorted(parquets):
        size_mb = os.path.getsize(f'data/{f}') / (1024**2)
        print(f'  {f}: {size_mb:.1f} MB')
else:
    print('No data yet. Run 00_data_acquisition.py first.')

print()
print('Environment ready.')

---

## Step 6: Run Topology Computation (the heavy step)

This runs persistent homology across all 28 CPC section pairs in sliding 5-year windows (1985\u20132023). Uses the **flagser** backend (directed flag complex) which is memory-efficient \u2014 works directly on sparse adjacency, no distance matrix needed.

**Memory fixes applied:**
- `gc.collect()` between every window and every pair
- Explicit `del` of intermediate objects
- Flagser avoids the distance matrix that caused the previous 170 GB OOM

**Expected time:** ~1\u20132 hours on A100, ~2\u20134 hours on T4  
**Expected memory:** ~20\u201340 GB peak (well within A100's 80\u2013170 GB)

In [ ]:
import subprocess
import psutil
import os

# Verify data files exist before starting
data_dir = '/content/the-shape-of-discovery/data'
required = ['patents.parquet', 'citations.parquet', 'cpc_map.parquet']
missing = [f for f in required if not os.path.exists(os.path.join(data_dir, f))]

if missing:
    print(f'ERROR: Missing data files: {missing}')
    print('Upload these files to Google Drive under shape-of-discovery/data/')
    print('Then re-run Step 4 (Restore from Drive) above.')
else:
    ram_gb = psutil.virtual_memory().total / (1024**3)
    print(f'Data files present. RAM available: {ram_gb:.1f} GB')
    print(f'Running topology computation with flagser backend...')
    print(f'This will take 1-4 hours. Progress updates below.\n')

    # Run the topology script \u2014 flagser backend, memory-safe defaults
    result = subprocess.run(
        [
            'python', 'scripts/run_topology.py',
            '--backend', 'flagser',
            '--max-nodes', '30000',
        ],
        cwd='/content/the-shape-of-discovery',
    )

    if result.returncode == 0:
        cache_dir = os.path.join(data_dir, 'topology_cache')
        if os.path.exists(cache_dir):
            cached = [f for f in os.listdir(cache_dir) if f.endswith('.pkl')]
            print(f'\nDone! {len(cached)} topology cache files generated.')
        else:
            print('\nWarning: topology_cache directory not found.')
    else:
        print(f'\nScript exited with code {result.returncode}. Check output above.')

### (Optional) Also run ripser backend for comparison
Only run this if flagser completed successfully and you have time/credits remaining. The ripser backend uses more memory (distance matrix computation). If it OOMs, reduce `--max-nodes` to 15000.

In [ ]:
# OPTIONAL: Run ripser backend (skip if short on time/memory)
# result = subprocess.run(
#     [
#         'python', 'scripts/run_topology.py',
#         '--backend', 'ripser',
#         '--max-nodes', '20000',  # Lower limit for ripser's distance matrix
#     ],
#     cwd='/content/the-shape-of-discovery',
# )
print('Ripser backend is commented out. Uncomment and run if needed.')

---

## Save results back to Drive
Run this cell **after** completing the topology computation (or any notebook) to persist results.

In [ ]:
import shutil
import glob
import os

drive_data = '/content/drive/MyDrive/shape-of-discovery/data'
drive_figs = '/content/drive/MyDrive/shape-of-discovery/figures'

# Save data (including topology_cache/)
saved = 0
for f in glob.glob('data/*'):
    if f.endswith('.gitkeep'):
        continue
    dest = os.path.join(drive_data, os.path.basename(f))
    if os.path.isdir(f):
        if os.path.exists(dest):
            shutil.rmtree(dest)
        shutil.copytree(f, dest)
    else:
        shutil.copy2(f, dest)
    saved += 1

# Save figures
for f in glob.glob('figures/*'):
    if f.endswith('.gitkeep'):
        continue
    shutil.copy2(f, os.path.join(drive_figs, os.path.basename(f)))
    saved += 1

print(f'Saved {saved} files to Google Drive.')
print('Your results will survive session disconnects.')

### Download topology cache locally
After the computation completes, run this to download the results as a zip file to your local machine.

In [ ]:
import shutil
import os

cache_dir = 'data/topology_cache'
if os.path.exists(cache_dir) and os.listdir(cache_dir):
    shutil.make_archive('/content/topology_cache', 'zip', cache_dir)
    from google.colab import files
    files.download('/content/topology_cache.zip')
    size_mb = os.path.getsize('/content/topology_cache.zip') / 1024 / 1024
    print(f'Downloading topology_cache.zip ({size_mb:.1f} MB)')
else:
    print('No topology cache found. Run Step 6 first.')

---

## Workflow Reminder

1. **Start of session:** Run cells 1\u20135 above (mount \u2192 clone \u2192 install \u2192 restore \u2192 verify)
2. **Run topology computation:** Step 6 (the heavy step \u2014 only needs to run once)
3. **Save results:** Run the \"Save results\" cell to persist to Drive
4. **Download locally:** Run the \"Download topology cache\" cell
5. **Next session:** Steps 1\u20135 again \u2014 your data restores automatically from Drive

### Running other notebooks in Colab

Option A \u2014 Open directly:
```
File \u2192 Open notebook \u2192 GitHub tab \u2192 paste repo URL \u2192 select notebook
```

Option B \u2014 Run from this notebook:
```python
# For the data acquisition script:
!python 00_data_acquisition.py

# For Jupyter notebooks:
!jupyter nbconvert --execute --to notebook --inplace 01_patent_atlas.ipynb
```

Option C \u2014 Convert to Colab-native:
Upload the .ipynb files directly to Colab and run interactively.